In [ ]:
# NOTEBOOK 05: PREPARE DASHBOARD VIEWS FOR TABLEAU
#
# Pre-aggregates the outputs of Notebooks 03 and 04 into small,purpose-built "view CSVs" that map directly to specific Tableau
# visualizations. Each chart on the dashboard connects to ONE simple CSV, so the Tableau build is mostly drag-and-drop.
#
# All internal tags (e.g. 'annex_iii_restricted', 'endocrine_disruptor') are converted to consumer-friendly labels (e.g. 'Restricted Ingredient',
# 'Hormone Disruption Risk') so the dashboard displays plain language.
#
# Page 1 (Awareness - "What"):      3 final views + 1 optional view
# Page 2 (Knowledge - "So What"):   3 view CSVs
# Page 3 (Decision - "Now What"):   2 final views + 1 optional view

import os

import pandas as pd


# ----------------------------------------------------------
# STEP 0: SETUP
# ----------------------------------------------------------

RISK_SCORING_DIR = '../data/risk_scoring'
OUTPUT_PATH      = '../data/dashboard_views'
os.makedirs(OUTPUT_PATH, exist_ok=True)

SCORES_FILE          = os.path.join(RISK_SCORING_DIR,
                                    'product_risk_scores.csv')
LONG_FILE            = os.path.join(RISK_SCORING_DIR,
                                    'product_ingredient_long.csv')
INGR_SUMMARY_FILE    = os.path.join(RISK_SCORING_DIR,
                                    'ingredient_risk_summary.csv')
INGR_TAGS_FILE       = os.path.join(RISK_SCORING_DIR,
                                    'ingredient_tags.csv')
LONG_TAGGED_FILE     = os.path.join(RISK_SCORING_DIR,
                                    'product_ingredient_long_tagged.csv')
CONCERN_SUMMARY_FILE = os.path.join(RISK_SCORING_DIR,
                                    'product_concern_summary.csv')

print("STEP 0: Setup")
print()
print(f"  {'File':<45} {'Status':>10}")
print(f"  {'----':<45} {'------':>10}")
for f in [SCORES_FILE, LONG_FILE, INGR_SUMMARY_FILE,
          INGR_TAGS_FILE, LONG_TAGGED_FILE, CONCERN_SUMMARY_FILE]:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f"  {os.path.basename(f):<45} {status:>10}")


# ----------------------------------------------------------
# CONSUMER-FRIENDLY LABEL MAPPINGS
# ----------------------------------------------------------
# Internal tags are machine-readable identifiers used in the
# data pipeline. These mappings convert them to plain language
# that cosmetic consumers can understand at a glance.

CONCERN_LABELS = {
    'eu_banned':              'EU-Banned Ingredient',
    'annex_iii_restricted':   'Restricted Ingredient',
    'fragrance_allergen':     'Fragrance Allergen',
    'sensitizer':             'Allergy Risk',
    'endocrine_disruptor':    'Hormone Disruption Risk',
    'carcinogen_concern':     'Cancer Concern',
    'reproductive_toxin':     'Reproductive Health Concern',
    'formaldehyde_releaser':  'Formaldehyde Risk',
    'nitrosamine_concern':    'Chemical Contamination Risk',
    'photosensitizer':        'Sun Sensitivity Risk',
    'microplastic':           'Microplastic',
    'manufacturing_impurity': 'Production Contamination Risk',
    'irritant':               'Skin Irritation Risk',
    'drying':                 'Drying Effect',
    'environmental_concern':  'Environmental Concern',
}

FUNCTION_LABELS = {
    'uv_filter':             'UV Protection',
    'preservative':          'Preservative',
    'surfactant':            'Cleanser / Foaming Agent',
    'emulsifier':            'Texture Blender',
    'colorant':              'Color / Pigment',
    'antioxidant':           'Antioxidant',
    'thickener':             'Thickener',
    'humectant':             'Moisture Attractor',
    'moisturizer':           'Moisturizer',
    'emollient':             'Skin Softener',
    'absorbent':             'Oil Absorber',
    'chelator':              'Stability Agent',
    'film_former':           'Film Former',
    'soothing':              'Soothing Agent',
    'wax':                   'Wax / Structure',
    'skin_barrier':          'Skin Barrier Support',
    'conditioner':           'Hair Conditioner',
    'amino_acid':            'Amino Acid',
    'peptide':               'Peptide (Anti-aging)',
    'vitamin':               'Vitamin',
    'silicone':              'Silicone',
    'fatty_acid':            'Fatty Acid',
    'plant_extract':         'Plant Extract / Oil',
    'ph_adjuster':           'pH Balancer',
    'solvent':               'Solvent',
    'penetration_enhancer':  'Absorption Enhancer',
    'exfoliant':             'Exfoliant',
    'polymer':               'Synthetic Polymer',
    'mineral':               'Mineral',
    'salt':                  'Salt',
    'other':                 'Other',
}

RISK_LEVEL_LABELS = {
    'Low':         'Low Risk',
    'Low-Medium':  'Low-Moderate Risk',
    'Medium':      'Moderate Risk',
    'Medium-High': 'Elevated Risk',
    'High':        'High Risk',
}

COVERAGE_LABELS = {
    'High':   'Well Analyzed',
    'Medium': 'Partially Analyzed',
    'Low':    'Limited Analysis',
}

# Traffic-light color mapping (for Tableau conditional formatting)
RISK_TO_COLOR = {
    'Low':         'Green',
    'Low-Medium':  'Yellow-Green',
    'Medium':      'Yellow',
    'Medium-High': 'Orange',
    'High':        'Red',
}

LABEL_TO_COLOR = {
    'Low Risk':                   'Green',
    'Low Risk (Limited Data)':    'Light Green',
    'Medium Risk':                'Yellow',
    'Medium Risk (Limited Data)': 'Light Yellow',
    'High Risk':                  'Red',
    'High Risk (Limited Data)':   'Light Red',
    'Insufficient Data':          'Grey',
}


STEP 0: Setup

  File                                              Status
  ----                                              ------
  product_risk_scores.csv                               OK
  product_ingredient_long.csv                           OK
  ingredient_risk_summary.csv                           OK
  ingredient_tags.csv                                   OK
  product_ingredient_long_tagged.csv                    OK
  product_concern_summary.csv                           OK


In [2]:
# ----------------------------------------------------------
# STEP 1: LOAD INPUTS
# ----------------------------------------------------------

print()
print("STEP 1: Load Inputs")
print()

scores          = pd.read_csv(SCORES_FILE)
long_df         = pd.read_csv(LONG_FILE)
ingr_summary    = pd.read_csv(INGR_SUMMARY_FILE)
ingr_tags       = pd.read_csv(INGR_TAGS_FILE)
long_tagged     = pd.read_csv(LONG_TAGGED_FILE)
concern_summary = pd.read_csv(CONCERN_SUMMARY_FILE)

print(f"  {'product_risk_scores':<35} {len(scores):>10,} rows")
print(f"  {'product_ingredient_long':<35} {len(long_df):>10,} rows")
print(f"  {'ingredient_risk_summary':<35} {len(ingr_summary):>10,} rows")
print(f"  {'ingredient_tags':<35} {len(ingr_tags):>10,} rows")
print(f"  {'product_ingredient_long_tagged':<35} {len(long_tagged):>10,} rows")
print(f"  {'product_concern_summary':<35} {len(concern_summary):>10,} rows")



STEP 1: Load Inputs

  product_risk_scores                      7,544 rows
  product_ingredient_long                227,213 rows
  ingredient_risk_summary                    400 rows
  ingredient_tags                            400 rows
  product_ingredient_long_tagged         227,213 rows
  product_concern_summary                  7,544 rows


In [3]:
# ==========================================================
# PAGE 1: AWARENESS / EXPOSURE CONTEXT ("THE WHAT")
# ==========================================================

print()
print("=" * 60)
print("PAGE 1: AWARENESS (Exposure Context)")
print("=" * 60)


# ----------------------------------------------------------
# 1A. KPI TILES
# ----------------------------------------------------------

print()
print("STEP 1A: KPI Tiles")

total_products = len(concern_summary)
n_any_concern  = (concern_summary['total_concern_flags'] > 0).sum()
n_eu_banned    = (concern_summary.get('n_eu_banned',
                                       pd.Series([0])) > 0).sum()
n_allergen     = (concern_summary.get('n_fragrance_allergen',
                                       pd.Series([0])) > 0).sum()

kpi_tiles = pd.DataFrame([
    {'metric': 'Total Products Analyzed',
     'count': total_products,
     'percentage': None,
     'caption': 'Sephora products with available ingredient data (March 2023)'},
    {'metric': 'Products with Any Concern',
     'count': int(n_any_concern),
     'percentage': round(n_any_concern / total_products * 100, 1),
     'caption': f'{n_any_concern:,} of {total_products:,} products'},
    {'metric': 'Products with EU-Banned Ingredient',
     'count': int(n_eu_banned),
     'percentage': round(n_eu_banned / total_products * 100, 1),
     'caption': f'{n_eu_banned:,} products contain ingredients banned in EU'},
    {'metric': 'Products with Fragrance Allergen',
     'count': int(n_allergen),
     'percentage': round(n_allergen / total_products * 100, 1),
     'caption': f'{n_allergen:,} products contain documented allergens'},
])

out = os.path.join(OUTPUT_PATH, 'view_p1_kpi_tiles.csv')
kpi_tiles.to_csv(out, index=False)
print(f"  Saved: {os.path.basename(out)} ({len(kpi_tiles)} rows)")
print()
print("  Use these numbers in your hardcoded Tableau text boxes:")
print()
for _, row in kpi_tiles.iterrows():
    if pd.notna(row['percentage']):
        print(f"    {row['metric']:<40} {row['percentage']:.1f}%  "
              f"({int(row['count']):,} products)")
    else:
        print(f"    {row['metric']:<40} {int(row['count']):,}")



PAGE 1: AWARENESS (Exposure Context)

STEP 1A: KPI Tiles
  Saved: view_p1_kpi_tiles.csv (4 rows)

  Use these numbers in your hardcoded Tableau text boxes:

    Total Products Analyzed                  7,544
    Products with Any Concern                89.9%  (6,785 products)
    Products with EU-Banned Ingredient       5.5%  (412 products)
    Products with Fragrance Allergen         59.0%  (4,450 products)


In [4]:
# ----------------------------------------------------------
# 1B. CONCERN PREVALENCE (horizontal bar chart)
# ----------------------------------------------------------

print()
print("STEP 1B: Concern Prevalence Bar Chart")

concern_cols = [c for c in concern_summary.columns if c.startswith('n_')]
prevalence_rows = []
for c in concern_cols:
    tag_name = c.replace('n_', '')
    n_affected = (concern_summary[c] > 0).sum()
    prevalence_rows.append({
        'concern_type':       tag_name,
        'concern_type_label': CONCERN_LABELS.get(
            tag_name, tag_name.replace('_', ' ').title()
        ),
        'products_affected':  n_affected,
        'total_products':     total_products,
        'percentage':         round(n_affected / total_products * 100, 2),
    })

concern_prevalence = (
    pd.DataFrame(prevalence_rows)
      .sort_values('percentage', ascending=False)
      .reset_index(drop=True)
)

out = os.path.join(OUTPUT_PATH, 'view_p1_concern_prevalence.csv')
concern_prevalence.to_csv(out, index=False)
print(f"  Saved: {os.path.basename(out)} ({len(concern_prevalence)} rows)")
print(concern_prevalence[['concern_type_label', 'products_affected',
                           'percentage']].head(10).to_string(index=False))



STEP 1B: Concern Prevalence Bar Chart
  Saved: view_p1_concern_prevalence.csv (14 rows)
           concern_type_label  products_affected  percentage
         Skin Irritation Risk               6169       81.77
        Restricted Ingredient               4805       63.69
           Fragrance Allergen               4450       58.99
                 Allergy Risk               3733       49.48
                Drying Effect               2228       29.53
      Hormone Disruption Risk               1412       18.72
  Chemical Contamination Risk                635        8.42
Production Contamination Risk                488        6.47
  Reproductive Health Concern                485        6.43
         EU-Banned Ingredient                412        5.46


In [5]:
# ----------------------------------------------------------
# 1C. CATEGORY CONCERN INTENSITY (bar / heatmap)
# ----------------------------------------------------------

print()
print("STEP 1C: Category Concern Intensity")

category_intensity = (
    concern_summary
        .groupby('primary_category')
        .agg(n_products=('product_id', 'count'),
             avg_concerns_per_product=('total_concern_flags', 'mean'),
             avg_risk_score=('weighted_score', 'mean'),
             pct_with_any_concern=('total_concern_flags',
                                    lambda x: (x > 0).mean() * 100))
        .assign(
            avg_concerns_per_product=lambda d:
                d['avg_concerns_per_product'].round(2),
            avg_risk_score=lambda d: d['avg_risk_score'].round(3),
            pct_with_any_concern=lambda d:
                d['pct_with_any_concern'].round(1),
        )
        .reset_index()
        .sort_values('avg_concerns_per_product', ascending=False)
)

out = os.path.join(OUTPUT_PATH, 'view_p1_category_intensity.csv')
category_intensity.to_csv(out, index=False)
print(f"  Saved: {os.path.basename(out)} ({len(category_intensity)} rows)")
print(category_intensity.to_string(index=False))



STEP 1C: Category Concern Intensity
  Saved: view_p1_category_intensity.csv (8 rows)
primary_category  n_products  avg_concerns_per_product  avg_risk_score  pct_with_any_concern
       Fragrance        1255                     19.19           1.657                  94.7
            Hair        1271                     12.79           1.242                  96.4
     Bath & Body         374                      9.76           1.205                  86.9
             Men          59                      9.32           1.179                  98.3
       Mini Size         266                      8.09           1.168                  94.7
        Skincare        2286                      7.21           1.109                  89.2
          Makeup        2031                      4.01           1.067                  83.5
 Tools & Brushes           2                      1.50           1.250                  50.0


In [6]:
# ----------------------------------------------------------
# 1D. OPTIONAL PRODUCT RISK SCATTERPLOT
# Retained as a supporting analysis view; it is not required in the
# final dashboard layout.
# ----------------------------------------------------------

print()
print("STEP 1D: Product Risk Scatterplot")

scatter = scores[[
    'product_id', 'product_name', 'brand_name',
    'primary_category', 'weighted_score', 'final_risk_label',
    'risk_category', 'coverage_pct', 'coverage_flag',
    'has_high_risk_warning', 'high_risk_ingredients_list',
    'total_ingredients', 'classified_ingredients',
    'low_count', 'low_medium_count', 'medium_count',
    'medium_high_count', 'high_count',
]].copy()

scatter['n_risky_ingredients'] = (
    scatter['medium_count']
    + scatter['medium_high_count']
    + scatter['high_count']
)

# Consumer-friendly labels
scatter['coverage_label'] = scatter['coverage_flag'].map(COVERAGE_LABELS)
scatter['badge_color']    = scatter['final_risk_label'].map(LABEL_TO_COLOR)

out = os.path.join(OUTPUT_PATH, 'view_p1_risk_scatterplot.csv')
scatter.to_csv(out, index=False)
print(f"  Saved: {os.path.basename(out)} ({len(scatter):,} rows)")
print(f"  Range: n_risky_ingredients 0-{scatter['n_risky_ingredients'].max()}, "
      f"weighted_score "
      f"{scatter['weighted_score'].min()}-"
      f"{scatter['weighted_score'].max()}")



STEP 1D: Product Risk Scatterplot
  Saved: view_p1_risk_scatterplot.csv (7,544 rows)
  Range: n_risky_ingredients 0-18, weighted_score 1.0-2.0


In [7]:
# ==========================================================
# PAGE 2: KNOWLEDGE / RISK TRANSLATION ("THE SO WHAT")
# ==========================================================

print()
print("=" * 60)
print("PAGE 2: KNOWLEDGE (Risk Translation)")
print("=" * 60)


# ----------------------------------------------------------
# 2A. INGREDIENT DIRECTORY (searchable table)
# ----------------------------------------------------------

print()
print("STEP 2A: Ingredient Directory")

ingr_dir = ingr_tags.merge(
    ingr_summary[['canonical_name', 'products_containing',
                  'top_primary_categories']],
    on='canonical_name', how='left'
)

ingr_dir['products_containing'] = (
    ingr_dir['products_containing'].fillna(0).astype(int)
)
ingr_dir['top_primary_categories'] = (
    ingr_dir['top_primary_categories'].fillna('')
)

# Consumer-friendly labels
ingr_dir['color_group']       = ingr_dir['risk_level'].map(RISK_TO_COLOR)
ingr_dir['risk_level_label']  = ingr_dir['risk_level'].map(RISK_LEVEL_LABELS)
ingr_dir['function_tag_label'] = ingr_dir['function_tag'].map(FUNCTION_LABELS)

# Convert concern_tag_list to consumer labels
def convert_concern_labels(tag_string):
    if not isinstance(tag_string, str) or tag_string.strip() == '':
        return ''
    tags = [t.strip() for t in tag_string.split(';')]
    labels = [CONCERN_LABELS.get(t, t.replace('_', ' ').title())
              for t in tags]
    return '; '.join(labels)

ingr_dir['concern_labels'] = (
    ingr_dir['concern_tag_list'].apply(convert_concern_labels)
)

ingr_dir = ingr_dir.sort_values(
    ['sub_weight', 'products_containing'],
    ascending=[False, False]
).reset_index(drop=True)

out = os.path.join(OUTPUT_PATH, 'view_p2_ingredient_directory.csv')
ingr_dir.to_csv(out, index=False)
print(f"  Saved: {os.path.basename(out)} ({len(ingr_dir):,} rows)")
print(ingr_dir[['canonical_name', 'risk_level_label', 'function_tag_label',
                 'products_containing']].head(8).to_string(index=False))



PAGE 2: KNOWLEDGE (Risk Translation)

STEP 2A: Ingredient Directory
  Saved: view_p2_ingredient_directory.csv (400 rows)
                              canonical_name risk_level_label function_tag_label  products_containing
                 butylphenyl methylpropional        High Risk              Other                  387
hydroxyisohexyl 3-cyclohexene carboxaldehyde        High Risk              Other                   88
                          parfum (fragrance)    Moderate Risk              Other                 3566
                                    geraniol    Moderate Risk              Other                 1527
                                      citral    Moderate Risk              Other                 1360
                              benzyl alcohol    Moderate Risk              Other                 1265
                                         bht    Moderate Risk        Antioxidant                 1015
                           benzyl salicylate    Moderate Risk 

In [8]:
# ----------------------------------------------------------
# 2B. INGREDIENT BY CONCERN TAG (exploded)
# ----------------------------------------------------------

print()
print("STEP 2B: Ingredient by Concern Tag (exploded)")

ingr_concern = ingr_tags[
    ingr_tags['concern_tag_list'].fillna('') != ''
].copy()
ingr_concern['concern_tag'] = (
    ingr_concern['concern_tag_list'].str.split('; ')
)
ingr_concern = ingr_concern.explode('concern_tag')
ingr_concern = ingr_concern[
    ingr_concern['concern_tag'].notna()
    & (ingr_concern['concern_tag'] != '')
]

ingr_concern = ingr_concern.merge(
    ingr_summary[['canonical_name', 'products_containing']],
    on='canonical_name', how='left'
)
ingr_concern['products_containing'] = (
    ingr_concern['products_containing'].fillna(0).astype(int)
)

# Consumer-friendly labels
ingr_concern['concern_tag_label'] = (
    ingr_concern['concern_tag'].map(CONCERN_LABELS)
)
ingr_concern['risk_level_label'] = (
    ingr_concern['risk_level'].map(RISK_LEVEL_LABELS)
)

ingr_concern_out = ingr_concern[[
    'canonical_name', 'concern_tag', 'concern_tag_label',
    'risk_level', 'risk_level_label', 'sub_weight',
    'products_containing', 'health_concern',
]].sort_values(
    ['concern_tag', 'sub_weight', 'products_containing'],
    ascending=[True, False, False]
).reset_index(drop=True)

out = os.path.join(OUTPUT_PATH, 'view_p2_ingredient_by_concern.csv')
ingr_concern_out.to_csv(out, index=False)
print(f"  Saved: {os.path.basename(out)} "
      f"({len(ingr_concern_out):,} rows, "
      f"{ingr_concern_out['concern_tag'].nunique()} unique concerns)")



STEP 2B: Ingredient by Concern Tag (exploded)
  Saved: view_p2_ingredient_by_concern.csv (116 rows, 14 unique concerns)


In [9]:
# ----------------------------------------------------------
# 2C. INGREDIENT CATEGORY USAGE (for viz-in-tooltip)
# ----------------------------------------------------------

print()
print("STEP 2C: Ingredient Category Usage")

ingr_in_products = long_tagged[
    long_tagged['risk_level'] != 'Unclassified'
].merge(
    scores[['product_id', 'primary_category']],
    on='product_id', how='left'
)

ingr_category = (
    ingr_in_products
        .groupby(['canonical_name', 'primary_category'])
        .size()
        .reset_index(name='n_products_in_category')
)

ingr_category = (
    ingr_category
        .sort_values(['canonical_name', 'n_products_in_category'],
                     ascending=[True, False])
        .groupby('canonical_name')
        .head(5)
        .reset_index(drop=True)
)

out = os.path.join(OUTPUT_PATH, 'view_p2_ingredient_category_usage.csv')
ingr_category.to_csv(out, index=False)
print(f"  Saved: {os.path.basename(out)} "
      f"({len(ingr_category):,} rows)")



STEP 2C: Ingredient Category Usage
  Saved: view_p2_ingredient_category_usage.csv (1,940 rows)


In [10]:
# ==========================================================
# PAGE 3: DECISION / SAFETY DECISION ("THE NOW WHAT")
# ==========================================================

print()
print("=" * 60)
print("PAGE 3: DECISION (Safety Decision)")
print("=" * 60)


# ----------------------------------------------------------
# 3A. PRODUCT DIRECTORY (searchable table)
# ----------------------------------------------------------

print()
print("STEP 3A: Product Directory")

product_dir = scores[[
    'product_id', 'product_name', 'brand_name',
    'primary_category', 'secondary_category', 'tertiary_category',
    'price_usd', 'rating',
    'weighted_score', 'risk_category', 'final_risk_label',
    'coverage_pct', 'coverage_flag',
    'total_ingredients', 'classified_ingredients',
    'has_high_risk_warning', 'high_risk_ingredients_list',
    'low_count', 'low_medium_count', 'medium_count',
    'medium_high_count', 'high_count',
]].copy()

# Attach key concern counts
key_concerns = ['n_eu_banned', 'n_fragrance_allergen',
                'n_endocrine_disruptor', 'n_formaldehyde_releaser',
                'n_carcinogen_concern', 'n_reproductive_toxin']
existing_key = [c for c in key_concerns if c in concern_summary.columns]
product_dir = product_dir.merge(
    concern_summary[['product_id'] + existing_key + ['total_concern_flags']],
    on='product_id', how='left'
)

# Consumer-friendly labels
product_dir['badge_color']    = product_dir['final_risk_label'].map(LABEL_TO_COLOR)
product_dir['coverage_label'] = product_dir['coverage_flag'].map(COVERAGE_LABELS)

out = os.path.join(OUTPUT_PATH, 'view_p3_product_directory.csv')
product_dir.to_csv(out, index=False)
print(f"  Saved: {os.path.basename(out)} ({len(product_dir):,} rows)")



PAGE 3: DECISION (Safety Decision)

STEP 3A: Product Directory
  Saved: view_p3_product_directory.csv (7,544 rows)


In [11]:
# ----------------------------------------------------------
# 3B. PRODUCT INGREDIENT DETAIL (drill-down)
# ----------------------------------------------------------

print()
print("STEP 3B: Product Ingredient Detail")

product_ingr_detail = long_tagged[[
    'product_id', 'raw_ingredient', 'canonical_name',
    'risk_level', 'sub_weight',
    'health_concern',
    'function_tag', 'concern_tag_list',
]].copy()

product_ingr_detail = product_ingr_detail.merge(
    scores[['product_id', 'product_name', 'brand_name',
            'primary_category']],
    on='product_id', how='left'
)

# Consumer-friendly labels
product_ingr_detail['color_group'] = (
    product_ingr_detail['risk_level'].map(RISK_TO_COLOR).fillna('Grey')
)
product_ingr_detail['risk_level_label'] = (
    product_ingr_detail['risk_level'].map(RISK_LEVEL_LABELS).fillna('Not Classified')
)
product_ingr_detail['function_tag_label'] = (
    product_ingr_detail['function_tag'].map(FUNCTION_LABELS).fillna('')
)
product_ingr_detail['concern_labels'] = (
    product_ingr_detail['concern_tag_list'].apply(convert_concern_labels)
)

out = os.path.join(OUTPUT_PATH, 'view_p3_product_ingredient_detail.csv')
product_ingr_detail.to_csv(out, index=False)
size_mb = os.path.getsize(out) / (1024 * 1024)
print(f"  Saved: {os.path.basename(out)} "
      f"({len(product_ingr_detail):,} rows, {size_mb:.1f} MB)")



STEP 3B: Product Ingredient Detail
  Saved: view_p3_product_ingredient_detail.csv (227,213 rows, 40.2 MB)


In [12]:
# ----------------------------------------------------------
# 3C. OPTIONAL CATEGORY SCORE DISTRIBUTION (comparison context)
# Retained as a supporting analysis view; it is not required in the
# final dashboard layout.
# ----------------------------------------------------------

print()
print("STEP 3C: Category Score Distribution")

category_dist = scores[[
    'product_id', 'product_name', 'brand_name',
    'primary_category', 'weighted_score',
    'final_risk_label', 'coverage_pct',
]].copy()
category_dist = category_dist.dropna(subset=['weighted_score'])

cat_means = (
    category_dist.groupby('primary_category')['weighted_score']
        .agg(['mean', 'median'])
        .reset_index()
        .rename(columns={'mean': 'category_avg_score',
                         'median': 'category_median_score'})
)
category_dist = category_dist.merge(
    cat_means, on='primary_category', how='left'
)
category_dist['category_avg_score']    = category_dist['category_avg_score'].round(3)
category_dist['category_median_score'] = category_dist['category_median_score'].round(3)

# Consumer-friendly labels
category_dist['badge_color'] = category_dist['final_risk_label'].map(LABEL_TO_COLOR)

out = os.path.join(OUTPUT_PATH, 'view_p3_category_score_distribution.csv')
category_dist.to_csv(out, index=False)
print(f"  Saved: {os.path.basename(out)} "
      f"({len(category_dist):,} rows)")



STEP 3C: Category Score Distribution
  Saved: view_p3_category_score_distribution.csv (7,367 rows)


In [13]:
# ==========================================================
# STEP 99: SUMMARY
# ==========================================================

print()
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print()
print(f"  All view CSVs saved to: {OUTPUT_PATH}")
print()

output_files = sorted([
    f for f in os.listdir(OUTPUT_PATH) if f.endswith('.csv')
])
for f in output_files:
    path = os.path.join(OUTPUT_PATH, f)
    size_kb = os.path.getsize(path) / 1024
    n_rows = len(pd.read_csv(path, usecols=[0]))
    if size_kb > 1024:
        size_str = f"{size_kb/1024:.1f} MB"
    else:
        size_str = f"{size_kb:.0f} KB"
    print(f"    {f:<50} {n_rows:>7,} rows  ({size_str})")

print()
print("Each CSV maps to one or two Tableau worksheets.")
print("Connect them as separate data sources in Tableau Public.")
print()
print("Label columns to use in Tableau (consumer-facing):")
print("  - concern_type_label  (not concern_type)")
print("  - risk_level_label    (not risk_level)")
print("  - function_tag_label  (not function_tag)")
print("  - coverage_label      (not coverage_flag)")
print("  - concern_labels      (not concern_tag_list)")
print("  - badge_color         (for conditional color)")
print()
print("DONE.")



SUMMARY

  All view CSVs saved to: ../data/dashboard_views

    view_p1_category_intensity.csv                           8 rows  (0 KB)
    view_p1_concern_prevalence.csv                          14 rows  (1 KB)
    view_p1_kpi_tiles.csv                                    4 rows  (0 KB)
    view_p1_risk_scatterplot.csv                         7,544 rows  (1.1 MB)
    view_p2_ingredient_by_concern.csv                      116 rows  (17 KB)
    view_p2_ingredient_category_usage.csv                1,940 rows  (60 KB)
    view_p2_ingredient_directory.csv                       400 rows  (69 KB)
    view_p3_category_score_distribution.csv              7,367 rows  (808 KB)
    view_p3_product_directory.csv                        7,544 rows  (1.5 MB)
    view_p3_product_ingredient_detail.csv              227,213 rows  (40.2 MB)

Each CSV maps to one or two Tableau worksheets.
Connect them as separate data sources in Tableau Public.

Label columns to use in Tableau (consumer-facing):
  - conce